# Surgical Instrument Detector — YOLOv8 Training on Google Colab

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

Run each cell in order from top to bottom.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Change runtime type to T4 GPU.')

## 2. Install Dependencies

In [ ]:
!pip install ultralytics roboflow requests pyyaml --quiet

## 3. Download Dataset from Roboflow

Fill in your credentials in the cell below.

In [ ]:
import io, os, shutil, zipfile, requests, yaml

# ── YOUR CREDENTIALS ──────────────────────────────────────────────────────────
API_KEY        = "YOUR_API_KEY"          # e.g. "v8JIfO0qH6gkYiX1YBzD"
WORKSPACE_NAME = "YOUR_WORKSPACE"        # e.g. "surgical-instruments-recognition"
PROJECT_NAME   = "YOUR_PROJECT"          # e.g. "final-year-development-project-fnerv"
VERSION_NUMBER = 1
# ──────────────────────────────────────────────────────────────────────────────

DATASET_DIR = "/content/dataset"

# Clean slate
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR)

# Get export URL
api_url = f"https://api.roboflow.com/{WORKSPACE_NAME}/{PROJECT_NAME}/{VERSION_NUMBER}/yolov8?api_key={API_KEY}"
resp = requests.get(api_url, timeout=30)
resp.raise_for_status()
info = resp.json()
download_url = info.get("export", {}).get("link")

if not download_url:
    # Trigger export generation
    import time
    print("Export not ready, triggering generation...")
    gen_url = f"https://api.roboflow.com/{WORKSPACE_NAME}/{PROJECT_NAME}/{VERSION_NUMBER}/yolov8/export?api_key={API_KEY}"
    requests.post(gen_url, timeout=30)
    for i in range(30):
        time.sleep(10)
        print(f"  Waiting... ({(i+1)*10}s)")
        resp = requests.get(api_url, timeout=30)
        download_url = resp.json().get("export", {}).get("link")
        if download_url:
            break
    else:
        raise RuntimeError("Export did not become ready after 5 minutes.")

# Download and extract zip
print("Downloading dataset zip...")
zip_resp = requests.get(download_url, timeout=300)
zip_resp.raise_for_status()
print(f"Downloaded {len(zip_resp.content)/1024:.0f} KB")

with zipfile.ZipFile(io.BytesIO(zip_resp.content)) as zf:
    print(f"Zip contains {len(zf.namelist())} files")
    zf.extractall(DATASET_DIR)

print("\nDataset directory contents:")
for entry in sorted(os.listdir(DATASET_DIR)):
    full = os.path.join(DATASET_DIR, entry)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  {entry}/  ({n} items)")
    else:
        print(f"  {entry}")

## 4. Fix data.yaml Paths

In [ ]:
yaml_path = os.path.join(DATASET_DIR, "data.yaml")

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

print("Original data.yaml:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

split_map = {"train": "train", "val": "valid", "test": "test"}
changed = False
for key, split in split_map.items():
    if key not in cfg:
        continue
    raw = cfg[key]
    if os.path.isabs(raw):
        candidate = raw
    else:
        candidate = os.path.normpath(os.path.join(DATASET_DIR, raw))
    if not os.path.isdir(candidate):
        fallback = os.path.join(DATASET_DIR, split, "images")
        if os.path.isdir(fallback):
            candidate = fallback
    if candidate != raw:
        cfg[key] = candidate
        changed = True

if changed:
    with open(yaml_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("\nFixed data.yaml:")
    for k, v in cfg.items():
        print(f"  {k}: {v}")
else:
    print("\nPaths already correct.")

## 5. Train YOLOv8n

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Training on: {'GPU - ' + torch.cuda.get_device_name(0) if device == 0 else 'CPU (no GPU found — change runtime to T4 GPU)'}")

model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name="surgical_instrument_detector",
    device=device,
    conf=0.5,
    plots=True,
)

print("\nTraining complete!")
print(f"Best weights: {results.save_dir}/weights/best.pt")

## 6. Evaluate on Test Set

In [ ]:
best_weights = str(results.save_dir) + "/weights/best.pt"
trained_model = YOLO(best_weights)

metrics = trained_model.val(data=yaml_path, split="test")
print(f"\nmAP50   : {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 7. Download Trained Weights

This downloads `best.pt` to your local machine.

In [ ]:
from google.colab import files
files.download(best_weights)
print("best.pt downloaded.")